In [6]:
import os
os.chdir(r"c:\Users\PLP\Documents\dsf-final-project-setc")
print(os.getcwd())

c:\Users\PLP\Documents\dsf-final-project-setc


In [10]:
import joblib
for p in ["application/models/logreg_70k.joblib",
          "application/models/logreg_200k.joblib"]:
    m = joblib.load(p)
    print("=" * 60)
    print(p)
    print("features:", m["features"])
    print("n_features:", len(m["features"]))
    print("feature_set_name:", m["feature_set_name"])

application/models/logreg_70k.joblib
features: ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
n_features: 16
feature_set_name: shared_16_features
application/models/logreg_200k.joblib
features: ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
n_features: 16
feature_set_name: shared_16_features


In [13]:
for f in ["application/app.py", "application/risk_model.py"]:
    print("=" * 60, f)
    with open(f, encoding="utf-8", errors="replace") as fh:
        for i, line in enumerate(fh, 1):
            if any(k in line for k in ["CholCheck", "AnyHealthcare", "features", "FEATURES"]):
                print(f"{i}: {line.rstrip()}")

============================================================ application/app.py
6: The questions are built from the features stored inside the saved model file, so the app keeps
24:     "CholCheck": ("Cholesterol check in the last 5 years", "Have you had your cholesterol checked in the past 5 years?"),
32:     "AnyHealthcare": ("Has health insurance or coverage", "Do you have any kind of health care coverage, including insurance, prepaid plans or government plans?"),
40:     ("Medical history", ["HighBP", "HighChol", "CholCheck", "Stroke", "HeartDiseaseorAttack"]),
43:     ("Healthcare access", ["AnyHealthcare", "NoDocbcCost"]),
87: FEATURES = bundle["features"]
88: unknown = [f for f in FEATURES if f not in KNOWN]
90:     st.error(f"The model uses features this app has no question for: {unknown}")
95:     st.write(f"**Questions asked:** {len(FEATURES)}")
118: if "BMI" in FEATURES:
210:         feats = [f for f in feats if f in FEATURES]
================================================

In [14]:
with open("application/risk_model.py", encoding="utf-8", errors="replace") as fh:
    for i, line in enumerate(fh, 1):
        low = line.lower()
        if any(k in low for k in ["bmi", "clip", "lower risk", "higher risk", "label", "risk_level"]):
            print(f"{i}: {line.rstrip()}")

43: BMI_MIN, BMI_MAX = 12, 98      # range of BMI values the model saw in training
51: def age_group_label(code: int) -> str:
58: def compute_bmi(height_cm: float, weight_kg: float) -> float:
62: def bmi_to_code(bmi: float) -> int:
63:     """BMI as a whole number, kept inside the range seen in training."""
64:     return int(round(min(max(bmi, BMI_MIN), BMI_MAX)))
67: def bmi_category(bmi: float) -> str:
68:     if bmi < 18.5: return "Underweight"
69:     if bmi < 25: return "Healthy weight"
70:     if bmi < 30: return "Overweight"


In [15]:
with open("application/risk_model.py", encoding="utf-8", errors="replace") as fh:
    lines = fh.readlines()
for i in range(50, 70):
    print(f"{i+1}: {lines[i].rstrip()}")

51: def age_group_label(code: int) -> str:
52:     if code == 13:
53:         return "80 or older"
54:     lo = 18 if code == 1 else AGE_UPPER_BOUNDS[code - 2]
55:     return f"{lo}-{AGE_UPPER_BOUNDS[code - 1] - 1}"
56: 
57: 
58: def compute_bmi(height_cm: float, weight_kg: float) -> float:
59:     return weight_kg / (height_cm / 100) ** 2
60: 
61: 
62: def bmi_to_code(bmi: float) -> int:
63:     """BMI as a whole number, kept inside the range seen in training."""
64:     return int(round(min(max(bmi, BMI_MIN), BMI_MAX)))
65: 
66: 
67: def bmi_category(bmi: float) -> str:
68:     if bmi < 18.5: return "Underweight"
69:     if bmi < 25: return "Healthy weight"
70:     if bmi < 30: return "Overweight"


In [16]:
with open("application/risk_model.py", encoding="utf-8", errors="replace") as fh:
    for i, line in enumerate(fh, 1):
        if "compute_bmi" in line:
            print(f"{i}: {line.rstrip()}")

58: def compute_bmi(height_cm: float, weight_kg: float) -> float:


In [17]:
with open("application/app.py", encoding="utf-8", errors="replace") as fh:
    for i, line in enumerate(fh, 1):
        if "compute_bmi" in line or "height" in line.lower() or "weight" in line.lower():
            print(f"{i}: {line.rstrip()}")

116: # Units sit outside the form so that the height and weight boxes change as soon as you switch.
119:     units = st.radio("Units for height and weight", ["Metric (cm, kg)", "Imperial (ft/in, lb)"], horizontal=True)
159: def ask_height_weight():
163:             h = st.number_input("Height (cm)", min_value=100.0, max_value=250.0, value=None, step=1.0,
166:             w = st.number_input("Weight (kg)", min_value=30.0, max_value=300.0, value=None, step=0.5,
168:         height_cm = h
169:         weight_kg = w
173:             ft = f1.number_input("Height (ft)", min_value=3, max_value=8, value=None, step=1, key="q_h_ft", placeholder="5")
176:             lb = st.number_input("Weight (lb)", min_value=66.0, max_value=660.0, value=None, step=1.0,
178:         height_cm = None if ft is None or inch is None else rm.feet_inches_to_cm(ft, inch)
179:         weight_kg = None if lb is None else rm.pounds_to_kg(lb)
181:     if height_cm is None or weight_kg is None:
182:         record("BMI", 

In [18]:
import joblib
m = joblib.load("application/models/logreg_200k.joblib")
for k, v in m.items():
    if k == "model":
        print(k, "->", type(v).__name__)
    elif k == "training_data":
        print(k, "->", v if not hasattr(v, "__len__") or len(str(v)) < 500 else f"{type(v).__name__} len={len(v)}")
    else:
        print(k, "->", v)

model -> Pipeline
features -> ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
threshold -> 0.3760557416737564
target_recall -> 0.85
feature_set_name -> shared_16_features
random_state -> 42
sklearn_version -> 1.9.1
training_data -> 200k: 100,000 no diabetes (real) + 100,000 diabetes (real + synthetic)


In [22]:
import pandas as pd
tr = pd.read_csv("data/05_cmb_train_test_for_models/diabetes_model_train.csv")
te = pd.read_csv("data/05_cmb_train_test_for_models/diabetes_model_test.csv")
print("train shape:", tr.shape, "test shape:", te.shape)
print("train target counts:\n", tr.iloc[:, -1].value_counts() if tr.columns[-1] not in tr.columns[:1] else "check cols")
print("train columns:", list(tr.columns))
print("test columns:", list(te.columns))
print("train target counts:\n", tr["Diabetes_binary"].value_counts())
print("test target counts:\n", te["Diabetes_binary"].value_counts())

train shape: (56155, 22) test shape: (14039, 22)
train target counts:
 Income
8    14937
7     9180
6     8422
5     6620
4     5580
3     4661
2     3760
1     2995
Name: count, dtype: int64
train columns: ['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
test columns: ['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
train target counts:
 Diabetes_binary
1    28078
0    28077
Name: count, dtype: int64
test target counts:
 Diabetes_binary
0    7020
1    7019
Name: count, dtype: int64


In [23]:
import pandas as pd
raw = pd.read_csv("data/01_original/diabetes_binary_health_indicators_BRFSS2015.csv")
print("raw shape:", raw.shape)
print("raw columns:", list(raw.columns)[:5], "...", list(raw.columns)[-3:])
print("Diabetes_binary counts:\n", raw["Diabetes_binary"].value_counts())

raw shape: (253680, 22)
raw columns: ['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI'] ... ['Age', 'Education', 'Income']
Diabetes_binary counts:
 Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64
